In [ ]:
# === [TFM_newcodes] redireccion de figuras a la carpeta ./figures/ ===
# Celda anadida automaticamente: cualquier figura guardada con un nombre
# de fichero simple (sin carpeta) se guarda dentro de ./figures/.
# Las rutas que ya incluyen una carpeta (p.ej. 'figures/...') se respetan.
import os as _os
import matplotlib.pyplot as _plt
from matplotlib.figure import Figure as _Figure
_os.makedirs('figures', exist_ok=True)
_os.makedirs('Data', exist_ok=True)  # CSVs de resultados circadianos
def _fig_redirect(fname):
    try:
        p = _os.fspath(fname)
    except TypeError:
        return fname  # objeto tipo fichero/buffer: no tocar
    if _os.path.dirname(p) == '':
        p = _os.path.join('figures', p)
    d = _os.path.dirname(p)
    if d:
        _os.makedirs(d, exist_ok=True)
    return p
if not getattr(_plt.savefig, '_tfm_patched', False):
    _orig_plt_savefig = _plt.savefig
    _orig_fig_savefig = _Figure.savefig
    def _plt_savefig(fname, *a, **k):
        return _orig_plt_savefig(_fig_redirect(fname), *a, **k)
    _plt_savefig._tfm_patched = True
    def _fig_savefig(self, fname, *a, **k):
        return _orig_fig_savefig(self, _fig_redirect(fname), *a, **k)
    _plt.savefig = _plt_savefig
    _Figure.savefig = _fig_savefig
savefig = _plt.savefig  # por si se usa savefig(...) directamente


# CNHPP failure — 3×3 results panel

Builds a 3×3 figure to show that the cascading non-homogeneous Poisson process (CNHPP) does **not** explain the data.

- **Columns** = datasets: (1) DANA all, (2) RENFE 2024, (3) RENFE 2026.
- **Row 1**: empirical CCDF vs CNHPP simulation (selected user of each column).
- **Row 2**: empirical CCDF vs week-reshuffled sequence (same user).
- **Row 3**: distribution of the area statistic `A_best`, original vs reshuffled, across all users (Wilcoxon test).

Data are read from the raw `.json` datasets and the saved CNHPP result CSVs (model + reshuffle).


In [27]:
import json, ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon


## Configuration

Adjust paths if your layout differs. Datasets in `../../Datasets`, result CSVs in this folder.


In [28]:
DATASETS = '../../Datasets'
RES = 'Data'

COLUMNS = [
    dict(name='DANA (all)',
         json=f'{DATASETS}/voluntariosdanavalencia_old.json',
         date_min=None, date_max=None,
         results=f'{RES}/circadian_cycles_all_results.csv',
         reshuffle=f'{RES}/circadian_cycles_all_reshuffle_results.csv'),
    dict(name='RENFE 2024',
         json=f'{DATASETS}/vagarenfe.json',
         date_min='2024-01-01', date_max='2024-12-31',
         results=f'{RES}/circadian_cycles_vagarenfe_2024_results.csv',
         reshuffle=f'{RES}/circadian_cycles_vagarenfe_2024_reshuffle_results.csv'),
    dict(name='RENFE 2026',
         json=f'{DATASETS}/vagarenfe.json',
         date_min='2026-01-15', date_max=None,
         results=f'{RES}/circadian_cycles_vagarenfe_2026_results.csv',
         reshuffle=f'{RES}/circadian_cycles_vagarenfe_2026_reshuffle_results.csv'),
]

# Display user per column (rows 1-2). None = auto-pick a high-activity user with a fitted result.
USER_IDS = [1762777403, 1848901170, 145800528]
SEED = 42


## Helper functions


In [29]:
def parse_array(s):
    cleaned = str(s).strip().replace('\n', ' ')
    if ',' not in cleaned:
        cleaned = cleaned.replace('[', '').replace(']', '')
        return np.array([float(x) for x in cleaned.split()])
    return np.array(ast.literal_eval(cleaned))

def normalize(df):
    df['sender_id'] = pd.to_numeric(df['sender_id'], errors='coerce').astype('Int64')
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    if df['date'].dt.tz is not None:
        df['date'] = df['date'].dt.tz_localize(None)
    return df.dropna(subset=['sender_id', 'date']).sort_values('date')

def ccdf(x):
    x = np.sort(np.asarray(x, float)); x = x[np.isfinite(x) & (x > 0)]
    y = 1.0 - np.arange(1, len(x) + 1) / len(x)
    return x, y


In [30]:
def simulate_cascading_nhpp(rho_a, Nw, pd_t, pw_t, p_Na, total_hours, n_samples=None, seed=0):
    """Cascading NHPP (Malmgren et al. 2008): primary rate Nw*pd(t)*pw(t),
    secondary HPP at rate rho_a within each active interval, Na ~ p_Na."""
    rng = np.random.default_rng(seed)
    pd_t = np.asarray(pd_t, float); pd_t = pd_t / pd_t.sum()
    pw_t = np.asarray(pw_t, float); pw_t = pw_t / pw_t.sum()
    p_Na = np.asarray(p_Na, float); p_Na = p_Na / p_Na.sum()
    mean_Na = np.dot(np.arange(len(p_Na)), p_Na)
    sim_hours = total_hours
    if n_samples is not None and Nw > 0:
        expected = Nw * (total_hours / (7 * 24)) * (1 + mean_Na)
        if expected < n_samples:
            sim_hours = total_hours * (n_samples / max(1, expected)) * 1.5
    n_weeks = sim_hours / (7 * 24)
    joint = np.outer(pw_t, pd_t).flatten(); joint /= joint.sum()
    n_bursts = rng.poisson(max(1.0, Nw * n_weeks))
    week_nums = rng.uniform(0, n_weeks, n_bursts)
    slots = rng.choice(168, size=n_bursts, p=joint)
    burst = (np.floor(week_nums) * 7 * 24 + (slots // 24) * 24 + (slots % 24) + rng.uniform(0, 1, n_bursts))
    burst = np.sort(burst); burst = burst[burst < sim_hours]
    if rho_a <= 0: rho_a = 1.0
    ev = []
    for bt in burst:
        ev.append(bt); Na = rng.choice(len(p_Na), p=p_Na); t = bt
        for _ in range(Na):
            t += rng.exponential(1.0 / rho_a); ev.append(t)
    res = np.sort(np.array(ev))
    return res[:n_samples] if n_samples is not None else res


In [31]:
def week_reshuffle(dates, seed=42):
    """Permute week index keeping (weekday, time-of-day). Returns (orig_iets_s, reshuffled_iets_s)."""
    rng = np.random.default_rng(seed)
    d = pd.Series(dates).dropna().sort_values().reset_index(drop=True)
    first_monday = d.iloc[0] - pd.Timedelta(days=d.iloc[0].weekday())
    week = ((d - first_monday).dt.total_seconds() // (7 * 86400)).astype(int).values
    wd  = d.dt.weekday.values
    tod = (d.dt.hour * 3600 + d.dt.minute * 60 + d.dt.second).values
    ots = d.values.astype('int64') / 1e9
    orig = np.diff(np.sort(ots)); orig = orig[orig > 0]
    perm = rng.permutation(week)
    nts = (perm * 7 * 86400 + wd * 86400 + tod).astype(float)
    nts = np.sort(nts); new = np.diff(nts); new = new[new > 0]
    return orig, new


## Build the 3×3 panel


In [32]:
import seaborn as sns
import matplotlib as mpl

sns.set_theme(style='ticks', context='paper')
mpl.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8,
    'legend.fontsize': 7, 'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'axes.spines.top': False, 'axes.spines.right': False,
})

C_EMP, C_MODEL, C_RESH = '#D55E00', '#56B4E9', '#009E73'  # Wong (2011) CB-safe

COL_HEADERS  = ['DANA', 'RENFE 2024', 'RENFE 2026']
PANEL_LABELS = [['(a)', '(b)', '(c)'],
                ['(d)', '(e)', '(f)'],
                ['(g)', '(h)', '(i)']]

fig, axes = plt.subplots(3, 3, figsize=(8.5, 5.0))
fig.subplots_adjust(hspace=0.72, wspace=0.38)

for c, cfg in enumerate(COLUMNS):
    df = normalize(pd.read_json(cfg['json']))
    if cfg['date_min']: df = df[df['date'] >= cfg['date_min']]
    if cfg['date_max']: df = df[df['date'] <= cfg['date_max']]
    resdf  = pd.read_csv(cfg['results']);   resdf['user_id']  = resdf['user_id'].astype('int64')
    reshdf = pd.read_csv(cfg['reshuffle']); reshdf['user_id'] = reshdf['user_id'].astype('int64')
    counts = df['sender_id'].value_counts()
    uid = USER_IDS[c]
    if uid is None:
        cand = sorted([u for u in resdf['user_id'] if u in counts.index], key=lambda u: counts[u], reverse=True)
        uid = cand[len(cand) // 2]
    print(f"{cfg['name']}: user {uid} ({int(counts.get(uid, 0))} messages)")
    row = resdf[resdf['user_id'] == uid].iloc[0]
    ev = df[df['sender_id'] == uid]['date'].sort_values()
    ev_h = (ev - ev.iloc[0]).dt.total_seconds().values / 3600
    emp_iets = np.diff(ev_h) * 3600
    sim = simulate_cascading_nhpp(row['rho_a'], row['Nw'], parse_array(row['pd_t']),
                                  parse_array(row['pw_t']), parse_array(row['p_Na']),
                                  total_hours=ev_h[-1], n_samples=len(ev_h), seed=SEED)
    sim_iets = np.diff(sim) * 3600

    # Row 0: Empirical vs CNHPP
    ax = axes[0, c]
    x, y = ccdf(emp_iets); ax.step(x, y, where='post', color=C_EMP,   lw=1.2, label='Empirical')
    x, y = ccdf(sim_iets); ax.step(x, y, where='post', color=C_MODEL, lw=1.2, label='CNHPP')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('IET (s)')
    ax.set_title(COL_HEADERS[c], fontsize=9, fontweight='bold', pad=14)
    ax.text(0.03, 1.12, PANEL_LABELS[0][c], transform=ax.transAxes,
            fontsize=8, va='bottom', ha='left', fontweight='bold')
    if c == 0:
        ax.set_ylabel('CCDF Empirical vs CNHPP')
        ax.legend(loc='lower left')

    # Row 1: Empirical vs reshuffled
    orig, resh = week_reshuffle(ev, seed=SEED)
    ax = axes[1, c]
    x, y = ccdf(orig); ax.step(x, y, where='post', color=C_EMP,  lw=1.2, label='Empirical')
    x, y = ccdf(resh); ax.step(x, y, where='post', color=C_RESH, lw=1.2, label='Week-reshuffled')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('IET (s)')
    ax.text(0.03, 1.08, PANEL_LABELS[1][c], transform=ax.transAxes,
            fontsize=8, va='bottom', ha='left', fontweight='bold')
    if c == 0:
        ax.set_ylabel('CCDF Empirical vs reshuffled')
        ax.legend(loc='lower left')

    # Row 2: A_best histogram
    merged = (resdf[['user_id', 'A_best']]
              .merge(reshdf[['user_id', 'A_best']], on='user_id', suffixes=('_orig', '_resh'))
              .dropna())
    ax = axes[2, c]
    ax.hist(merged['A_best_orig'], bins=20, alpha=0.6, color=C_EMP,   edgecolor='black', lw=0.4, label='Original')
    ax.hist(merged['A_best_resh'], bins=20, alpha=0.6, color=C_MODEL, edgecolor='black', lw=0.4, label='Reshuffled')
    ax.axvline(merged['A_best_orig'].median(), color=C_EMP,   ls='--', lw=1.0)
    ax.axvline(merged['A_best_resh'].median(), color=C_MODEL, ls='--', lw=1.0)
    try:
        stat, p = wilcoxon(merged['A_best_orig'], merged['A_best_resh'], alternative='greater')
        #ax.set_title(f'Wilcoxon: p = {p:.2g},  n = {len(merged)}', fontsize=7.5, pad=14)
    except Exception as e:
        #ax.set_title(f'Wilcoxon failed: {e}', fontsize=7, pad=14)
        pass
    ax.set_xlabel('A_best (lower = better fit)')
    ax.text(0.03, 1.08, PANEL_LABELS[2][c], transform=ax.transAxes,
            fontsize=8, va='bottom', ha='left', fontweight='bold')
    if c == 0:
        ax.set_ylabel('A_best orig vs reshuffled')
        ax.legend()

fig.savefig('260624_cnhpp_panel_paper.pdf', bbox_inches='tight')
plt.show()
print('Saved 260624_cnhpp_panel_paper.pdf')
